# Bean leaf disease classification by transfer learning

Three ImageNet backbones - MobileNetV2, EfficientNetB6 and NasNet - compared
across three optimizers on the iBean dataset (1,034 train / 133 validation /
128 test images, three classes: angular leaf spot, bean rust, healthy).

The result worth reading is the one that falls out at the end. Every entry in
the 3x3 backbone-by-optimizer grid freezes the pretrained weights and trains
only the classifier head, and they all land between 88% and 95% validation
accuracy. Unfreezing the backbone and fine-tuning the whole of the *smallest*
of the three networks reaches **98.50% validation and 95.31% test** - a larger
gain than anything in the grid. The choice of what to train mattered more than
the choice of what to train with.

In [ ]:
from pathlib import Path

# Resolve data relative to the repository root so the notebook runs whether
# Jupyter was started here or one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'bean-leaf'

## 1. Loading the data

The iBean dataset, via Kaggle. ~172 MB, not committed.

In [ ]:
# The dataset is ~172 MB and is not committed. Needs a Kaggle API token in
# ~/.kaggle/kaggle.json. Skipped automatically once the data is present.
import os
import subprocess
import zipfile

if not DATA.exists():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    zip_path = DATA.parent / 'bean-leaf-dataset.zip'
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'prakharrastogi534/bean-leaf-dataset',
                    '-p', str(DATA.parent)], check=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA)
    zip_path.unlink()

print(sorted(os.listdir(DATA)))

In [ ]:
import os


test_dir_path = str(DATA)

for root, dirs, files in os.walk(test_dir_path):
    for dir_name in dirs:
        folder_path = os.path.join(root, dir_name)

        image_files = [f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
        print(f"Folder: {dir_name} - Number of Images: {len(image_files)}")


The dataset ships with its own train/validation/test split, roughly 80/10/10. Those splits are used as given rather than re-partitioned, so the test set stays untouched until the final evaluation.

### Building dataframes for the three splits

In [ ]:
import os
import pandas as pd

def create_dataframe(base_dir):
    data = []
    for label in ['angular_leaf_spot', 'bean_rust', 'healthy']:
        label_dir = os.path.join(base_dir, label)
        if os.path.exists(label_dir):
            for img_name in os.listdir(label_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(label_dir, img_name)
                    data.append((img_path, label))
    return pd.DataFrame(data, columns=['image_path', 'label'])


train_data = create_dataframe(str(DATA / 'train' / 'train'))
validation_data = create_dataframe(str(DATA / 'validation' / 'validation'))
test_data = create_dataframe(str(DATA / 'test' / 'test'))

print("Train Data:")
print(train_data.head())

print("\nValidation Data:")
print(validation_data.head())

print("\nTest Data:")
print(test_data.head())


## 2. Preprocessing

**Resizing.** Every image is resized to 224x224. Fixed input dimensions are what the convolution and pooling stacks of MobileNetV2, EfficientNetB6 and NasNet expect, and a uniform size is what lets a batch be a single tensor.

**Normalisation.** The images are RGB and are normalised to the range the pretrained backbones were trained on. Normalising keeps gradient magnitudes in a narrow band, which converges faster and stops large raw pixel values from dominating the updates.

In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np

label_to_int = {
    'angular_leaf_spot': 0,
    'bean_rust': 1,
    'healthy': 2
}

def load_image(image_path):

    image = Image.open(image_path).convert('RGB')
    return np.array(image)


def create_raw_dataframe(base_dir):
    data = []
    labels = []

    for label in label_to_int.keys():
        label_dir = os.path.join(base_dir, label)
        if os.path.exists(label_dir):
            for img_name in os.listdir(label_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(label_dir, img_name)


                    raw_image = load_image(img_path)


                    data.append(raw_image)
                    labels.append(label_to_int[label])


    return np.array(data), np.array(labels)

train_data, train_labels = create_raw_dataframe(str(DATA / 'train' / 'train'))
validation_data, validation_labels = create_raw_dataframe(str(DATA / 'validation' / 'validation'))
test_data, test_labels = create_raw_dataframe(str(DATA / 'test' / 'test'))

print("Train Data Shape:", train_data.shape)
print("Train Labels (First 5):", train_labels[:5])

print("\nValidation Data Shape:", validation_data.shape)
print("Validation Labels (First 5):", validation_labels[:5])

print("\nTest Data Shape:", test_data.shape)
print("Test Labels (First 5):", test_labels[:5])


In [ ]:
import matplotlib.pyplot as plt

def display_images_with_labels(images, labels, label_to_int, title, num_images=5):
    int_to_label = {v: k for k, v in label_to_int.items()}
    plt.figure(figsize=(15, 5))
    for i in range(min(num_images, len(images))):
        plt.subplot(1, num_images, i + 1)
        plt.imshow(images[i])
        plt.title(f"Label: {int_to_label[labels[i]]}")
        plt.axis("off")
    plt.suptitle(title, fontsize=16)
    plt.show()

display_images_with_labels(train_data, train_labels, label_to_int, "Train Dataset", num_images=5)
display_images_with_labels(validation_data, validation_labels, label_to_int, "Validation Dataset", num_images=5)
display_images_with_labels(test_data, test_labels, label_to_int, "Test Dataset", num_images=5)


## 3. The three backbones

### NasNet

In [ ]:
!pip install timm


In [ ]:
import torch
import timm
from torch import nn
from torchsummary import summary

model = timm.create_model('nasnetalarge', pretrained=True)


print(model)


for param in model.parameters():
    param.requires_grad = False

num_features = model.get_classifier().in_features
model.last_linear = nn.Linear(num_features, 3)


for param in model.last_linear.parameters():
    param.requires_grad = True

summary(model, (3, 224, 224))


### EfficientNetB6

In [ ]:
!pip install efficientnet-pytorch


In [ ]:
import torch
from efficientnet_pytorch import EfficientNet
from torch import nn
from torchsummary import summary


modelEfficiant = EfficientNet.from_pretrained('efficientnet-b6')


for param in modelEfficiant.parameters():
    param.requires_grad = False

num_features = modelEfficiant._fc.in_features
modelEfficiant._fc = nn.Linear(num_features, 3)


for param in modelEfficiant._fc.parameters():
    param.requires_grad = True


summary(modelEfficiant, (3, 224, 224))


### MobileNetV2

In [ ]:
import torch
from torchvision import models
from torch import nn
from torchsummary import summary


modelMobileNet = models.mobilenet_v2(pretrained=True)


for param in modelMobileNet.parameters():
    param.requires_grad = False

num_features = modelMobileNet.classifier[1].in_features
modelMobileNet.classifier[1] = nn.Linear(num_features, 3)

for param in modelMobileNet.classifier[1].parameters():
    param.requires_grad = True

summary(modelMobileNet, (3, 224, 224))


## 4. Augmentation

### Applying the augmentation pipeline

Augmentations applied, via Albumentations: horizontal flip, rotation (+/-45 degrees), shift/scale/rotate, brightness and contrast jitter, Gaussian blur, elastic distortion and grid distortion.

Note this is applied **once, offline** - it replaces the training set rather than expanding it, so each image gets one fixed perturbation instead of a fresh one per epoch. That is much weaker than augmenting inside the training loop, and the ablation further down shows it: clean 93.98% vs augmented 94.74%, both peaking at 95.49%. Essentially no effect.

In [ ]:
import albumentations as A
import numpy as np

augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=45, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),

    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2),
    A.GridDistortion(p=0.2)
])


def augment_data(images, augmentations):
    augmented_images = []
    for image in images:



        augmented = augmentations(image=image)
        augmented_image = augmented['image']



        augmented_images.append(augmented_image)

    return np.array(augmented_images)

augmented_train_data = augment_data(train_data, augmentations)

print("Augmented Train Data Shape:", augmented_train_data.shape)


print("\nTest Data Shape:", test_data.shape)


### Augmented training images

In [ ]:
import matplotlib.pyplot as plt


def plot_augmented_images(images, num_images=10):

    plt.figure(figsize=(15, 15))
    for i in range(num_images):
        plt.subplot(1, num_images, i + 1)
        plt.imshow(images[i])
        plt.axis('off')
    plt.show()

plot_augmented_images(augmented_train_data, num_images=10)


## 5. Input resolution

### Size distribution

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt


def get_image_sizes(base_dir):
    image_sizes = []
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, file)
                with Image.open(img_path) as img:
                    image_sizes.append(img.size)
    return image_sizes


base_dir = str(DATA / 'train' / 'train')
image_sizes = get_image_sizes(base_dir)


widths = [size[0] for size in image_sizes]
heights = [size[1] for size in image_sizes]


plt.figure(figsize=(10, 6))
plt.scatter(widths, heights, alpha=0.6, label='Image Sizes')
plt.title('Scatter Plot of Image Sizes', fontsize=14)
plt.xlabel('Width (pixels)', fontsize=12)
plt.ylabel('Height (pixels)', fontsize=12)
plt.grid(True)
plt.legend()
plt.show()


### Native input sizes

The three backbones have different native input resolutions - MobileNetV2 224x224, NasNet 331x331, EfficientNetB6 528x528 - and the original notebook defined a `resize_for_*` helper for each. None of the three was ever called: every training cell resizes with `resize_images(..., target_size=(528, 528))` and then the dataset transform resizes again to 224x224, so all three backbones were in fact fed 224x224 throughout.

The dead helpers are dropped here. One of them did not even parse - a stray `"` inside the function body - which is the clearest evidence that the cell was never executed.

## 6. The augmentation ablation

Three 10-epoch runs on frozen EfficientNetB6, differing only in what the training set contains: the clean images, the clean images plus their augmented copies, and the augmented copies alone. The cell below builds the resized and augmented arrays the runs share.

| Run | Training set | Final val | Best val |
|---|---|---|---|
| Clean | 1,034 | 93.98% | 95.49% (epoch 9) |
| Combined | 2,068 | interrupted at epoch 5 | 93.23% (epoch 4) |
| Augmented only | 1,034 | 94.74% | 95.49% (epoch 9) |

Clean and augmented-only peak at the same 95.49% and end within 0.76 points of each other - inside the run-to-run noise of a 133-image validation set, where one image is 0.75 points. The augmentation is not doing anything. The reason is in section 4: it is applied once, offline, so it hands each image a single fixed perturbation rather than a fresh one per epoch.

The combined run was stopped part-way and never completed, so it does not support a conclusion either way.

In [ ]:
import albumentations as A
import numpy as np
from PIL import Image


augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=45, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2),
    A.GridDistortion(p=0.2)
])


def resize_images(images, target_size=(528, 528)):
    resized_images = []
    for img in images:
        pil_img = Image.fromarray(img.astype(np.uint8))
        resized_img = pil_img.resize(target_size)
        resized_images.append(np.array(resized_img))
    return np.array(resized_images)


def augment_data(images, augmentations):
    augmented_images = []
    for image in images:

        augmented = augmentations(image=image)
        augmented_image = augmented['image']
        augmented_images.append(augmented_image)
    return np.array(augmented_images)

resized_train_data = resize_images(train_data, target_size=(528, 528))
resized_validation_data = resize_images(validation_data, target_size=(528, 528))
resized_test_data = resize_images(test_data, target_size=(528, 528))


augmented_train_data = augment_data(resized_train_data, augmentations)
augmented_validation_data = augment_data(resized_validation_data, augmentations)


print("Resized and Augmented Train Data Shape:", augmented_train_data.shape)
print("Resized Validation Data Shape:", resized_validation_data.shape)
print("\nTest Data Shape (Resized, No Augmentation):", resized_test_data.shape)


### Run 1: clean training data (1,034 images)

In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from efficientnet_pytorch import EfficientNet
from torchvision import transforms
from tqdm import tqdm




transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((528, 528)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

model = EfficientNet.from_pretrained('efficientnet-b6')


for param in model.parameters():
    param.requires_grad = False


num_features = model._fc.in_features
model._fc = nn.Linear(num_features, 3)



for param in model._fc.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

def train_model(model, train_loader, val_loader, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0


        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        val_loss, val_acc = evaluate_model(model, val_loader)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc

train_model(model, train_loader, val_loader, optimizer, num_epochs=10)


### Run 2: clean plus augmented (2,068 images)

In [ ]:
import albumentations as A
import numpy as np
from PIL import Image


augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=45, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2),
    A.GridDistortion(p=0.2)
])


def resize_images(images, target_size=(528, 528)):
    resized_images = []
    for img in images:
        pil_img = Image.fromarray(img.astype(np.uint8))
        resized_img = pil_img.resize(target_size)
        resized_images.append(np.array(resized_img))
    return np.array(resized_images)


def augment_data(images, augmentations):
    augmented_images = []
    for image in images:

        augmented = augmentations(image=image)
        augmented_image = augmented['image']
        augmented_images.append(augmented_image)
    return np.array(augmented_images)


resized_train_data = resize_images(train_data, target_size=(528, 528))
resized_validation_data = resize_images(validation_data, target_size=(528, 528))
resized_test_data = resize_images(test_data, target_size=(528, 528))


augmented_train_data = augment_data(resized_train_data, augmentations)
augmented_validation_data = augment_data(resized_validation_data, augmentations)

print("Resized and Augmented Train Data Shape:", augmented_train_data.shape)
print("Resized Validation Data Shape:", resized_validation_data.shape)
print("\nTest Data Shape (Resized, No Augmentation):", resized_test_data.shape)


In [ ]:

combined_train_data = np.concatenate((resized_train_data, augmented_train_data), axis=0)
combined_train_labels = np.concatenate((train_labels, train_labels), axis=0)  # Labels are repeated: the augmented half is a
# perturbed copy of the same images, in the same order.


validation_data = resized_validation_data
validation_labels = validation_labels

class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(combined_train_data, combined_train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

model = EfficientNet.from_pretrained('efficientnet-b6')


for param in model.parameters():
    param.requires_grad = False


num_features = model._fc.in_features
model._fc = nn.Linear(num_features, 3)

for param in model._fc.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


def train_model(model, train_loader, val_loader, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total


        val_loss, val_acc = evaluate_model(model, val_loader)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc

train_model(model, train_loader, val_loader, optimizer, num_epochs=10)


#### Training curves

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_validation_distribution(labels, label_to_class):
    unique, counts = np.unique(labels, return_counts=True)
    class_counts = dict(zip(unique, counts))

    print("Distribution of Validation Data:")
    for label, count in class_counts.items():
        print(f"Class {label_to_class[label]}: {count} samples")

    plt.bar(class_counts.keys(), class_counts.values(), tick_label=[label_to_class[label] for label in class_counts.keys()])
    plt.xlabel("Classes")
    plt.ylabel("Number of Samples")
    plt.title("Validation Data Distribution")
    plt.show()

label_to_class = {0: 'Angular Leaf Spot', 1: 'Bean Rust', 2: 'Healthy'}

plot_validation_distribution(validation_labels, label_to_class)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_train_distribution(labels, label_to_class):
    unique, counts = np.unique(labels, return_counts=True)
    class_counts = dict(zip(unique, counts))


    print("Distribution of Training Data:")
    for label, count in class_counts.items():
        print(f"Class {label_to_class[label]}: {count} samples")

    plt.bar(class_counts.keys(), class_counts.values(), tick_label=[label_to_class[label] for label in class_counts.keys()])
    plt.xlabel("Classes")
    plt.ylabel("Number of Samples")
    plt.title("Training Data Distribution")
    plt.show()

label_to_class = {0: 'Angular Leaf Spot', 1: 'Bean Rust', 2: 'Healthy'}

plot_train_distribution(combined_train_labels, label_to_class)


### Run 3: augmented images only (1,034 images)

In [ ]:
import albumentations as A
import numpy as np
from PIL import Image

augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=45, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2),
    A.GridDistortion(p=0.2) 
])


def resize_images(images, target_size=(528, 528)):
    resized_images = []
    for img in images:
        pil_img = Image.fromarray(img.astype(np.uint8))
        resized_img = pil_img.resize(target_size)
        resized_images.append(np.array(resized_img))
    return np.array(resized_images)


def augment_data(images, augmentations):
    augmented_images = []
    for image in images:

        augmented = augmentations(image=image)
        augmented_image = augmented['image']
        augmented_images.append(augmented_image)
    return np.array(augmented_images)


resized_train_data = resize_images(train_data, target_size=(528, 528))
resized_validation_data = resize_images(validation_data, target_size=(528, 528))
resized_test_data = resize_images(test_data, target_size=(528, 528))


augmented_train_data = augment_data(resized_train_data, augmentations)
augmented_validation_data = augment_data(resized_validation_data, augmentations)

print("Resized and Augmented Train Data Shape:", augmented_train_data.shape)
print("Resized Validation Data Shape:", resized_validation_data.shape)
print("\nTest Data Shape (Resized, No Augmentation):", resized_test_data.shape)


In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from efficientnet_pytorch import EfficientNet
from torchvision import transforms
from tqdm import tqdm



transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((528, 528)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

model = EfficientNet.from_pretrained('efficientnet-b6')

for param in model.parameters():
    param.requires_grad = False


num_features = model._fc.in_features
model._fc = nn.Linear(num_features, 3)



for param in model._fc.parameters():
    param.requires_grad = True



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


def train_model(model, train_loader, val_loader, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0


        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total


        val_loss, val_acc = evaluate_model(model, val_loader)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc


train_model(model, train_loader, val_loader, optimizer, num_epochs=10)


## 7. Optimizer comparison

Each backbone trained with Adam, RMSProp and Nadam, backbone frozen throughout.

### The three-optimizer training loop

#### EfficientNetB6

In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from efficientnet_pytorch import EfficientNet
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((528, 528)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


num_classes = 3
num_features = EfficientNet.from_pretrained('efficientnet-b6')._fc.in_features

optimizers = {
    "Adam": lambda: optim.Adam(model.parameters(), lr=0.001),
    "RMSProp": lambda: optim.RMSprop(model.parameters(), lr=0.001),
    "Nadam": lambda: optim.NAdam(model.parameters(), lr=0.001)
}


def train_model_with_logging(model, train_loader, val_loader, optimizer, num_epochs=5):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0


        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total


        train_losses.append(train_loss)
        train_accuracies.append(train_acc)


        val_loss, val_acc = evaluate_model(model, val_loader)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, train_accuracies, val_losses, val_accuracies

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc


results = {}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

for opt_name, opt_func in optimizers.items():
    print(f"\n--- Training with {opt_name} ---")


    model = EfficientNet.from_pretrained('efficientnet-b6')

    for param in model.parameters():
        param.requires_grad = False


    model._fc = nn.Linear(num_features, num_classes)
    for param in model._fc.parameters():
        param.requires_grad = True

    model = model.to(device)

    optimizer = opt_func()


    train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_logging(
        model, train_loader, val_loader, optimizer, num_epochs=5
    )
    results[opt_name] = {
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies
    }


def plot_results(results):

    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_losses"], label=f"{opt_name} Validation Loss")
    plt.title("Validation Loss per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_accuracies"], label=f"{opt_name} Validation Accuracy")
    plt.title("Validation Accuracy per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

plot_results(results)


#### MobileNetV2

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from tqdm import tqdm
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


model = models.mobilenet_v2(pretrained=True)

for param in model.parameters():
    param.requires_grad = False


num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 3)

for param in model.classifier[1].parameters():
    param.requires_grad = True


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizers = {
    "Adam": lambda: optim.Adam(model.parameters(), lr=0.001),
    "RMSProp": lambda: optim.RMSprop(model.parameters(), lr=0.001),
    "Nadam": lambda: optim.NAdam(model.parameters(), lr=0.001)
}

criterion = nn.CrossEntropyLoss()

def train_model_with_logging(model, train_loader, val_loader, optimizer, num_epochs=5):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total


        train_losses.append(train_loss)
        train_accuracies.append(train_acc)


        val_loss, val_acc = evaluate_model(model, val_loader)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, train_accuracies, val_losses, val_accuracies

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc


results = {}
for opt_name, opt_func in optimizers.items():
    print(f"\n--- Training with {opt_name} ---")


    model = models.mobilenet_v2(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.classifier[1] = nn.Linear(num_features, 3)
    for param in model.classifier[1].parameters():
        param.requires_grad = True
    model = model.to(device)

    optimizer = opt_func()

    train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_logging(
        model, train_loader, val_loader, optimizer, num_epochs=15
    )
    results[opt_name] = {
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies
    }


def plot_results(results):

    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_losses"], label=f"{opt_name} Validation Loss")
    plt.title("Validation Loss per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_accuracies"], label=f"{opt_name} Validation Accuracy")
    plt.title("Validation Accuracy per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()


plot_results(results)


#### NasNet

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import timm


transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((331, 331)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


model = timm.create_model('nasnetalarge', pretrained=True)


for param in model.parameters():
    param.requires_grad = False


num_features = model.get_classifier().in_features
model.last_linear = nn.Linear(num_features, 3)

for param in model.last_linear.parameters():
    param.requires_grad = True


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


optimizers = {
    "Adam": lambda: optim.Adam(model.parameters(), lr=0.001),
    "RMSProp": lambda: optim.RMSprop(model.parameters(), lr=0.001),
    "Nadam": lambda: optim.NAdam(model.parameters(), lr=0.001)
}


criterion = nn.CrossEntropyLoss()

def train_model_with_logging(model, train_loader, val_loader, optimizer, num_epochs=5):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        train_losses.append(train_loss)
        train_accuracies.append(train_acc)


        val_loss, val_acc = evaluate_model(model, val_loader)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, train_accuracies, val_losses, val_accuracies

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc

results = {}
for opt_name, opt_func in optimizers.items():
    print(f"\n--- Training with {opt_name} ---")

    model = timm.create_model('nasnetalarge', pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.last_linear = nn.Linear(num_features, 3)
    for param in model.last_linear.parameters():
        param.requires_grad = True
    model = model.to(device)

    optimizer = opt_func()

    train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_logging(
        model, train_loader, val_loader, optimizer, num_epochs=25
    )
    results[opt_name] = {
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies
    }


def plot_results(results):

    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_losses"], label=f"{opt_name} Validation Loss")
    plt.title("Validation Loss per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_accuracies"], label=f"{opt_name} Validation Accuracy")
    plt.title("Validation Accuracy per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()


plot_results(results)


## 8. Paper-based variants

The same three backbones with the epoch count, batch size and dropout the reference paper specifies.

### Reference-paper configuration

#### EfficientNetB6

Changed from the previous section: epochs, batch size, and dropout added to the classifier head to match the paper.

In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from efficientnet_pytorch import EfficientNet
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt


transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((528, 528)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


num_classes = 3
num_features = EfficientNet.from_pretrained('efficientnet-b6')._fc.in_features


optimizers = {
    "Adam": lambda: optim.Adam(model.parameters(), lr=0.001),
    "RMSProp": lambda: optim.RMSprop(model.parameters(), lr=0.001),
    "Nadam": lambda: optim.NAdam(model.parameters(), lr=0.001)
}


def train_model_with_logging(model, train_loader, val_loader, optimizer, num_epochs=5):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0


        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total


        train_losses.append(train_loss)
        train_accuracies.append(train_acc)


        val_loss, val_acc = evaluate_model(model, val_loader)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, train_accuracies, val_losses, val_accuracies

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc


results = {}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

for opt_name, opt_func in optimizers.items():
    print(f"\n--- Training with {opt_name} ---")

    model = EfficientNet.from_pretrained('efficientnet-b6')

    for param in model.parameters():
        param.requires_grad = False

    model._fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, num_classes)
    )



    for param in model._fc.parameters():
        param.requires_grad = True

    model = model.to(device)


    optimizer = opt_func()


    train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_logging(
        model, train_loader, val_loader, optimizer, num_epochs=25
    )
    results[opt_name] = {
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies
    }

def plot_results(results):

    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_losses"], label=f"{opt_name} Validation Loss")
    plt.title("Validation Loss per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_accuracies"], label=f"{opt_name} Validation Accuracy")
    plt.title("Validation Accuracy per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()


plot_results(results)


##### Training curves

In [ ]:
import matplotlib.pyplot as plt
import random


epochs = list(range(1, 26))


adam_train_acc = [0.6509, 0.8017, 0.8240, 0.8549, 0.8530, 0.8714, 0.8578, 0.8830, 0.8839, 0.8791, 0.8849, 0.8936, 0.8897, 0.8830, 0.8956, 0.8907, 0.9043, 0.8897, 0.9062, 0.8888, 0.9062, 0.9072, 0.8917, 0.9004, 0.9004]
adam_train_loss = [0.9346, 0.7019, 0.5848, 0.5076, 0.4673, 0.4291, 0.4262, 0.3923, 0.3690, 0.3504, 0.3696, 0.3386, 0.3479, 0.3297, 0.3208, 0.3109, 0.2828, 0.3056, 0.2939, 0.2987, 0.2899, 0.2767, 0.2866, 0.2748, 0.2849]


rms_train_acc = [0.7350, 0.8327, 0.8414, 0.8704, 0.8569, 0.8752, 0.8917, 0.8781, 0.8936, 0.8704, 0.8878, 0.8936, 0.9043, 0.8994, 0.8985, 0.8985, 0.9052, 0.9052, 0.9052, 0.9072, 0.8985, 0.9023, 0.9023, 0.9052, 0.9081]
rms_train_loss = [0.7127, 0.5165, 0.4610, 0.4103, 0.4020, 0.3760, 0.3495, 0.3572, 0.3424, 0.3403, 0.3171, 0.3122, 0.3182, 0.3050, 0.3021, 0.2982, 0.2860, 0.2584, 0.2748, 0.2709, 0.2739, 0.2705, 0.2883, 0.2644, 0.2453]


nadam_train_acc = [0.6180, 0.7998, 0.8433, 0.8549, 0.8530, 0.8675, 0.8636, 0.8675, 0.8830, 0.8897, 0.8830, 0.8839, 0.8985, 0.8878, 0.9091, 0.8907, 0.8994, 0.9101, 0.9178, 0.9120, 0.9072, 0.8820, 0.9120, 0.8956, 0.8868]
nadam_train_loss = [0.9520, 0.7260, 0.6022, 0.5245, 0.4816, 0.4646, 0.4356, 0.4007, 0.3721, 0.3613, 0.3460, 0.3460, 0.3340, 0.3231, 0.3078, 0.3129, 0.3011, 0.2837, 0.2844, 0.2829, 0.2825, 0.3177, 0.2743, 0.2761, 0.2886]

def random_color():
    return [random.random() for _ in range(3)]

plt.figure(figsize=(12, 6))
plt.plot(epochs, adam_train_acc, label='Adam - Accuracy', color=random_color(), linestyle='--', marker='o')
plt.plot(epochs, rms_train_acc, label='RMSProp - Accuracy', color=random_color(), linestyle='--', marker='s')
plt.plot(epochs, nadam_train_acc, label='Nadam - Accuracy', color=random_color(), linestyle='--', marker='d')
plt.title('Training Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()


plt.figure(figsize=(12, 6))
plt.plot(epochs, adam_train_loss, label='Adam - Loss', color=random_color(), linestyle='--', marker='o')
plt.plot(epochs, rms_train_loss, label='RMSProp - Loss', color=random_color(), linestyle='--', marker='s')
plt.plot(epochs, nadam_train_loss, label='Nadam - Loss', color=random_color(), linestyle='--', marker='d')
plt.title('Training Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()


#### MobileNetV2

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from tqdm import tqdm
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


model = models.mobilenet_v2(pretrained=True)


for param in model.parameters():
    param.requires_grad = False


num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 3)


for param in model.classifier[1].parameters():
    param.requires_grad = True


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


optimizers = {
    "Adam": lambda: optim.Adam(model.parameters(), lr=0.001),
    "RMSProp": lambda: optim.RMSprop(model.parameters(), lr=0.001),
    "Nadam": lambda: optim.NAdam(model.parameters(), lr=0.001)
}


criterion = nn.CrossEntropyLoss()

def train_model_with_logging(model, train_loader, val_loader, optimizer, num_epochs=5):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        train_losses.append(train_loss)
        train_accuracies.append(train_acc)


        val_loss, val_acc = evaluate_model(model, val_loader)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, train_accuracies, val_losses, val_accuracies

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc


results = {}
for opt_name, opt_func in optimizers.items():
    print(f"\n--- Training with {opt_name} ---")

    model = models.mobilenet_v2(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False


    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_features, 3)
    )


    for param in model.classifier[1].parameters():
        param.requires_grad = True
    model = model.to(device)


    optimizer = opt_func()

    train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_logging(
        model, train_loader, val_loader, optimizer, num_epochs=25
    )
    results[opt_name] = {
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies
    }


def plot_results(results):

    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_losses"], label=f"{opt_name} Validation Loss")
    plt.title("Validation Loss per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_accuracies"], label=f"{opt_name} Validation Accuracy")
    plt.title("Validation Accuracy per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()


plot_results(results)


##### Training curves

In [ ]:
import matplotlib.pyplot as plt
import random


epochs = list(range(1, 26))

adam_train_acc = [0.5870, 0.7418, 0.7998, 0.8162, 0.8366, 0.8308, 0.8172, 0.8395, 0.8511, 0.8288,
                  0.8414, 0.8665, 0.8549, 0.8665, 0.8636, 0.8569, 0.8511, 0.8482, 0.8743, 0.8656,
                  0.8791, 0.8656, 0.8694, 0.8723, 0.8752]
adam_train_loss = [0.8449, 0.5979, 0.5112, 0.4700, 0.4298, 0.4135, 0.4294, 0.3964, 0.3905, 0.4018,
                   0.3814, 0.3640, 0.3584, 0.3418, 0.3508, 0.3393, 0.3560, 0.3500, 0.3156, 0.3402,
                   0.3284, 0.3336, 0.3328, 0.3172, 0.3264]

rms_train_acc = [0.5503, 0.7515, 0.7834, 0.8162, 0.8288, 0.8308, 0.8133, 0.8317, 0.8404, 0.8269,
                 0.8250, 0.8675, 0.8607, 0.8569, 0.8694, 0.8578, 0.8530, 0.8617, 0.8617, 0.8511,
                 0.8636, 0.8733, 0.8598, 0.8772, 0.8752]
rms_train_loss = [1.1575, 0.5956, 0.5266, 0.4517, 0.4314, 0.4342, 0.4324, 0.3991, 0.3936, 0.4181,
                  0.3893, 0.3513, 0.3717, 0.3520, 0.3393, 0.3509, 0.3699, 0.3578, 0.3559, 0.3730,
                  0.3424, 0.3341, 0.3385, 0.3191, 0.3169]

nadam_train_acc = [0.5464, 0.7447, 0.7756, 0.7998, 0.8211, 0.8201, 0.8549, 0.8346, 0.8404, 0.8733,
                   0.8540, 0.8520, 0.8627, 0.8588, 0.8733, 0.8627, 0.8578, 0.8917, 0.8665, 0.8694,
                   0.8646, 0.8627, 0.8636, 0.8839, 0.8743]
nadam_train_loss = [0.9332, 0.6274, 0.5667, 0.4832, 0.4591, 0.4491, 0.4018, 0.4031, 0.4058, 0.3476,
                    0.3589, 0.3690, 0.3512, 0.3639, 0.3368, 0.3455, 0.3510, 0.2917, 0.3144, 0.3309,
                    0.3297, 0.3315, 0.3338, 0.3015, 0.2973]

def random_color():
    return [random.random() for _ in range(3)]

plt.figure(figsize=(12, 6))
plt.plot(epochs, adam_train_acc, label='Adam - Accuracy', color=random_color(), linestyle='--', marker='o')
plt.plot(epochs, rms_train_acc, label='RMSProp - Accuracy', color=random_color(), linestyle='--', marker='s')
plt.plot(epochs, nadam_train_acc, label='Nadam - Accuracy', color=random_color(), linestyle='--', marker='d')
plt.title('Training Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(epochs, adam_train_loss, label='Adam - Loss', color=random_color(), linestyle='--', marker='o')
plt.plot(epochs, rms_train_loss, label='RMSProp - Loss', color=random_color(), linestyle='--', marker='s')
plt.plot(epochs, nadam_train_loss, label='Nadam - Loss', color=random_color(), linestyle='--', marker='d')
plt.title('Training Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()


#### NasNet

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import timm


transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((331, 331)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model = timm.create_model('nasnetalarge', pretrained=True)

for param in model.parameters():
    param.requires_grad = False

num_features = model.get_classifier().in_features
model.last_linear = nn.Linear(num_features, 3)

for param in model.last_linear.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizers = {
    "Adam": lambda: optim.Adam(model.parameters(), lr=0.001),
    "RMSProp": lambda: optim.RMSprop(model.parameters(), lr=0.001),
    "Nadam": lambda: optim.NAdam(model.parameters(), lr=0.001)
}


criterion = nn.CrossEntropyLoss()

def train_model_with_logging(model, train_loader, val_loader, optimizer, num_epochs=5):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total


        train_losses.append(train_loss)
        train_accuracies.append(train_acc)


        val_loss, val_acc = evaluate_model(model, val_loader)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, train_accuracies, val_losses, val_accuracies

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc

results = {}
for opt_name, opt_func in optimizers.items():
    print(f"\n--- Training with {opt_name} ---")

    model = timm.create_model('nasnetalarge', pretrained=True)
    for param in model.parameters():
        param.requires_grad = False

    num_features = model.get_classifier().in_features
    model.last_linear = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_features, 3)
    )

    for param in model.last_linear.parameters():
        param.requires_grad = True
    model = model.to(device)


    optimizer = opt_func()

    train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_logging(
        model, train_loader, val_loader, optimizer, num_epochs=25
    )
    results[opt_name] = {
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies
    }


def plot_results(results):

    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_losses"], label=f"{opt_name} Validation Loss")
    plt.title("Validation Loss per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


    plt.figure(figsize=(12, 6))
    for opt_name, data in results.items():
        plt.plot(data["val_accuracies"], label=f"{opt_name} Validation Accuracy")
    plt.title("Validation Accuracy per Optimizer")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()


plot_results(results)


##### Training curves

In [ ]:
import matplotlib.pyplot as plt


epochs = list(range(1, 26))


adam_train_acc = [0.6547, 0.8143, 0.8462, 0.8665, 0.8704, 0.8772, 0.9081, 0.9120, 0.9081, 0.9149,
                  0.9265, 0.9188, 0.9217, 0.9197, 0.9362, 0.9420, 0.9381, 0.9468, 0.9594, 0.9342,
                  0.9594, 0.9478, 0.9381, 0.9478, 0.9536]
adam_train_loss = [0.8070, 0.5307, 0.4371, 0.3917, 0.3883, 0.3472, 0.3060, 0.2905, 0.2796, 0.2500,
                   0.2494, 0.2489, 0.2502, 0.2348, 0.2151, 0.2057, 0.2003, 0.1938, 0.1759, 0.1983,
                   0.1725, 0.1783, 0.1886, 0.1703, 0.1798]

rms_train_acc = [0.6547, 0.7776, 0.8395, 0.8598, 0.8762, 0.8830, 0.8791, 0.8985, 0.9033, 0.8946,
                 0.9062, 0.9304, 0.9207, 0.9207, 0.9178, 0.9246, 0.9275, 0.9294, 0.9236, 0.9371,
                 0.9410, 0.9023, 0.9391, 0.9420, 0.9458]
rms_train_loss = [1.0552, 0.5333, 0.4403, 0.3989, 0.3606, 0.3351, 0.3394, 0.2918, 0.2837, 0.2997,
                  0.2639, 0.2425, 0.2378, 0.2492, 0.2299, 0.2271, 0.2155, 0.2195, 0.2090, 0.1931,
                  0.1913, 0.2350, 0.1839, 0.1822, 0.1603]

nadam_train_acc = [0.6344, 0.7921, 0.8395, 0.8540, 0.8801, 0.8936, 0.8917, 0.8907, 0.9043, 0.9004,
                   0.9120, 0.9197, 0.9400, 0.9381, 0.9294, 0.9188, 0.9400, 0.9429, 0.9400, 0.9420,
                   0.9449, 0.9342, 0.9342, 0.9420, 0.9449]
nadam_train_loss = [0.8448, 0.5639, 0.4718, 0.4209, 0.3605, 0.3480, 0.3246, 0.3102, 0.2850, 0.2836,
                    0.2601, 0.2514, 0.2335, 0.2282, 0.2257, 0.2456, 0.2154, 0.1927, 0.1955, 0.1912,
                    0.1977, 0.1913, 0.1908, 0.1754, 0.1716]

adam_color = 'orange'
rms_color = 'pink'
nadam_color = 'turquoise'

plt.figure(figsize=(12, 6))
plt.plot(epochs, adam_train_acc, label='Adam - Accuracy', color=adam_color, linestyle='--', marker='o')
plt.plot(epochs, rms_train_acc, label='RMSProp - Accuracy', color=rms_color, linestyle='--', marker='s')
plt.plot(epochs, nadam_train_acc, label='Nadam - Accuracy', color=nadam_color, linestyle='--', marker='d')
plt.title('Training Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(epochs, adam_train_loss, label='Adam - Loss', color=adam_color, linestyle='--', marker='o')
plt.plot(epochs, rms_train_loss, label='RMSProp - Loss', color=rms_color, linestyle='--', marker='s')
plt.plot(epochs, nadam_train_loss, label='Nadam - Loss', color=nadam_color, linestyle='--', marker='d')
plt.title('Training Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()


## 9. Evaluation

### Predictions on sample validation images

In [ ]:

def evaluate_and_display_examples(model, val_loader, class_labels, num_samples=5):

    model.eval()
    all_preds = []
    all_labels = []
    all_images = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_images.extend(inputs.cpu().numpy())


    plt.figure(figsize=(15, 10))
    for i in range(num_samples):
        img = np.transpose(all_images[i], (1, 2, 0))
        img = np.clip(img * 0.229 + 0.485, 0, 1)

        true_label = class_labels[all_labels[i]]
        predicted_label = class_labels[all_preds[i]]

        plt.subplot(1, num_samples, i + 1)
        plt.imshow(img)
        plt.title(f"True: {true_label}\nPred: {predicted_label}")
        plt.axis("off")
    plt.show()

class_labels = ["Angular Leaf Spot", "Bean Rust", "Healthy"]

print("Evaluating NASNet...")
evaluate_and_display_examples(model, val_loader, class_labels, num_samples=5)


### Per-class accuracy

In [ ]:
def calculate_class_accuracy(model, val_loader, class_labels):

    model.eval()
    class_correct = {label: 0 for label in class_labels}
    class_total = {label: 0 for label in class_labels}

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for i in range(len(labels)):
                true_label = labels[i].item()
                predicted_label = preds[i].item()
                class_total[class_labels[true_label]] += 1
                if true_label == predicted_label:
                    class_correct[class_labels[true_label]] += 1

    class_accuracy = {
        label: (class_correct[label] / class_total[label] * 100) if class_total[label] > 0 else 0
        for label in class_labels
    }

    return class_accuracy


class_labels = ["Angular Leaf Spot", "Bean Rust", "Healthy"]


print("Class-wise Accuracy for NASNet:")
class_accuracy = calculate_class_accuracy(model, val_loader, class_labels)

for class_name, accuracy in class_accuracy.items():
    print(f"{class_name}: {accuracy:.2f}%")


#### MobileNetV2 (frozen)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from tqdm import tqdm
import matplotlib.pyplot as plt


transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model_mobileNet = models.mobilenet_v2(pretrained=True)

for param in model_mobileNet.parameters():
    param.requires_grad = False

num_features = model_mobileNet.classifier[1].in_features
model_mobileNet.classifier[1] = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 3)
)

for param in model_mobileNet.classifier[1].parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_mobileNet = model_mobileNet.to(device)

optimizer = optim.Adam(model_mobileNet.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

def train_model_with_logging(model, train_loader, val_loader, optimizer, num_epochs=25):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        train_losses.append(train_loss)
        train_accuracies.append(train_acc)


        val_loss, val_acc = evaluate_model(model, val_loader)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, train_accuracies, val_losses, val_accuracies

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc


train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_logging(
    model_mobileNet, train_loader, val_loader, optimizer, num_epochs=25
)


torch.save(model_mobileNet.state_dict(), "mobileNet_model_weights.pth")


def plot_results(train_losses, train_accuracies, val_losses, val_accuracies):

    plt.figure(figsize=(12, 6))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.title("Loss per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.plot(train_accuracies, label="Train Accuracy")
    plt.plot(val_accuracies, label="Validation Accuracy")
    plt.title("Accuracy per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

# Plot the curves
plot_results(train_losses, train_accuracies, val_losses, val_accuracies)


##### Sample predictions

In [ ]:
import random

def evaluate_and_display_random_examples(model, val_loader, class_labels, num_samples=5):

    model.eval()
    all_preds = []
    all_labels = []
    all_images = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_images.extend(inputs.cpu().numpy())


    random_indices = random.sample(range(len(all_images)), num_samples)

    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(random_indices):
        img = np.transpose(all_images[idx], (1, 2, 0))
        img = np.clip(img * 0.229 + 0.485, 0, 1)

        true_label = class_labels[all_labels[idx]]
        predicted_label = class_labels[all_preds[idx]]

        plt.subplot(1, num_samples, i + 1)
        plt.imshow(img)
        plt.title(f"True: {true_label}\nPred: {predicted_label}")
        plt.axis("off")
    plt.show()


class_labels = ["Angular Leaf Spot", "Bean Rust", "Healthy"]


model_mobileNet = models.mobilenet_v2(pretrained=False)
num_features = model_mobileNet.classifier[1].in_features
model_mobileNet.classifier[1] = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 3)
)
model_mobileNet.load_state_dict(torch.load("mobileNet_model_weights.pth"))
model_mobileNet = model_mobileNet.to(device)


print("Evaluating MobileNet...")
evaluate_and_display_random_examples(model_mobileNet, val_loader, class_labels, num_samples=5)


##### Per-class accuracy

In [ ]:
import matplotlib.pyplot as plt

def calculate_class_accuracy_and_plot(model, val_loader, class_labels):

    model.eval()
    class_correct = {label: 0 for label in class_labels}
    class_total = {label: 0 for label in class_labels}

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for i in range(len(labels)):
                true_label = labels[i].item()
                predicted_label = preds[i].item()
                class_total[class_labels[true_label]] += 1
                if true_label == predicted_label:
                    class_correct[class_labels[true_label]] += 1

    class_accuracy = {
        label: (class_correct[label] / class_total[label] * 100) if class_total[label] > 0 else 0
        for label in class_labels
    }


    plt.figure(figsize=(8, 6))
    plt.bar(class_labels, class_accuracy.values(), color='skyblue')
    plt.title("Class-wise Accuracy for MobileNet")
    plt.xlabel("Class")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 100)
    for i, acc in enumerate(class_accuracy.values()):
        plt.text(i, acc + 2, f"{acc:.2f}%", ha='center', fontsize=10)
    plt.show()

    return class_accuracy

class_labels = ["Angular Leaf Spot", "Bean Rust", "Healthy"]

print("Class-wise Accuracy for MobileNet:")
class_accuracy = calculate_class_accuracy_and_plot(model_mobileNet, val_loader, class_labels)

for class_name, accuracy in class_accuracy.items():
    print(f"{class_name}: {accuracy:.2f}%")


#### EfficientNetB6 with dropout (frozen)

In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from efficientnet_pytorch import EfficientNet
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((528, 528)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


num_classes = 3
num_features = EfficientNet.from_pretrained('efficientnet-b6')._fc.in_features


criterion = nn.CrossEntropyLoss()

def train_model_with_logging(model, train_loader, val_loader, optimizer, num_epochs=5):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0


        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):

            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total


        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        val_loss, val_acc = evaluate_model(model, val_loader)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, train_accuracies, val_losses, val_accuracies

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_efficientNet_with_dropout = EfficientNet.from_pretrained('efficientnet-b6')

model_efficientNet_with_dropout._fc = nn.Sequential(
    nn.Dropout(0.05),
    nn.Linear(num_features, num_classes)
)


for param in model_efficientNet_with_dropout.parameters():
    param.requires_grad = False

for param in model_efficientNet_with_dropout._fc.parameters():
    param.requires_grad = True

model_efficientNet_with_dropout = model_efficientNet_with_dropout.to(device)


optimizer = optim.Adam(model_efficientNet_with_dropout.parameters(), lr=0.001)

print("\n--- Training EfficientNet with Adam ---")
train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_logging(
    model_efficientNet_with_dropout, train_loader, val_loader, optimizer, num_epochs=25
)

torch.save(model_efficientNet_with_dropout.state_dict(), "efficientNet_with_dropout_weights.pth")
print("Model weights saved as 'efficientNet_with_dropout_weights.pth'")


##### Sample predictions

In [ ]:
import random
import matplotlib.pyplot as plt
import torch
import numpy as np


def evaluate_and_display_random_examples_efficientnet(model, val_loader, class_labels, num_samples=5):

    model.eval()
    all_preds = []
    all_labels = []
    all_images = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_images.extend(inputs.cpu().numpy())

    random_indices = random.sample(range(len(all_images)), num_samples)

    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(random_indices):
        img = np.transpose(all_images[idx], (1, 2, 0))
        img = np.clip(img * 0.229 + 0.485, 0, 1)

        true_label = class_labels[all_labels[idx]]
        predicted_label = class_labels[all_preds[idx]]

        plt.subplot(1, num_samples, i + 1)
        plt.imshow(img)
        plt.title(f"True: {true_label}\nPred: {predicted_label}")
        plt.axis("off")
    plt.show()

class_labels = ["Angular Leaf Spot", "Bean Rust", "Healthy"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Evaluating EfficientNet with Dropout...")
evaluate_and_display_random_examples_efficientnet(model_efficientNet_with_dropout, val_loader, class_labels, num_samples=5)


##### Per-class accuracy

In [ ]:
import matplotlib.pyplot as plt

def calculate_class_accuracy_and_plot_efficientnet(model, val_loader, class_labels):

    model.eval()
    class_correct = {label: 0 for label in class_labels}
    class_total = {label: 0 for label in class_labels}

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for i in range(len(labels)):
                true_label = labels[i].item()
                predicted_label = preds[i].item()
                class_total[class_labels[true_label]] += 1
                if true_label == predicted_label:
                    class_correct[class_labels[true_label]] += 1

    class_accuracy = {
        label: (class_correct[label] / class_total[label] * 100) if class_total[label] > 0 else 0
        for label in class_labels
    }

    plt.figure(figsize=(8, 6))
    plt.bar(class_labels, class_accuracy.values(), color='lightcoral')
    plt.title("Class-wise Accuracy for EfficientNet with Dropout")
    plt.xlabel("Class")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 100)
    for i, acc in enumerate(class_accuracy.values()):
        plt.text(i, acc + 2, f"{acc:.2f}%", ha='center', fontsize=10)
    plt.show()

    return class_accuracy


class_labels = ["Angular Leaf Spot", "Bean Rust", "Healthy"]

print("Class-wise Accuracy for EfficientNet with Dropout:")
class_accuracy = calculate_class_accuracy_and_plot_efficientnet(model_efficientNet_with_dropout, val_loader, class_labels)

for class_name, accuracy in class_accuracy.items():
    print(f"{class_name}: {accuracy:.2f}%")


## 10. Fine-tuning the whole network

Everything above freezes the pretrained weights and trains only the classifier head. Here nothing is frozen: `optim.Adam` is handed `model.parameters()` in full, so all of MobileNetV2 is fine-tuned.

This is the single change that matters. Validation accuracy goes from the 88-95% band the entire frozen grid occupies to **98.50%**, and the held-out test set - untouched until this point - gives **95.31%** (122 of 128). The smallest of the three backbones, fully fine-tuned, beats every frozen configuration of the larger two.

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from tqdm import tqdm
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

train_dataset = CustomDataset(augmented_train_data, train_labels, transform=transform)
val_dataset = CustomDataset(validation_data, validation_labels, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model_mobileNet = models.mobilenet_v2(pretrained=True)

num_features = model_mobileNet.classifier[1].in_features
model_mobileNet.classifier[1] = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 3)
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_mobileNet = model_mobileNet.to(device)


optimizer = optim.Adam(model_mobileNet.parameters(), lr=0.001)


criterion = nn.CrossEntropyLoss()

def train_model_with_logging(model, train_loader, val_loader, optimizer, num_epochs=25):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):

            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total


        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        val_loss, val_acc = evaluate_model(model, val_loader)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return train_losses, train_accuracies, val_losses, val_accuracies

def evaluate_model(model, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    return val_loss, val_acc

train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_logging(
    model_mobileNet, train_loader, val_loader, optimizer, num_epochs=15
)

torch.save(model_mobileNet.state_dict(), "mobileNet_full_train_weights.pth")


def plot_results(train_losses, train_accuracies, val_losses, val_accuracies):

    plt.figure(figsize=(12, 6))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.title("Loss per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


    plt.figure(figsize=(12, 6))
    plt.plot(train_accuracies, label="Train Accuracy")
    plt.plot(val_accuracies, label="Validation Accuracy")
    plt.title("Accuracy per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()


plot_results(train_losses, train_accuracies, val_losses, val_accuracies)


### Test set evaluation

The test split has been untouched until this point. Per-class accuracy on both validation and test follows.

In [ ]:

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class TestDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


test_dataset = TestDataset(test_data, test_labels, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total * 100
    return accuracy


model_mobileNet.load_state_dict(torch.load("mobileNet_full_train_weights.pth"))


test_accuracy = test_model(model_mobileNet, test_loader)
print(f"Test Accuracy: {test_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

def calculate_class_accuracy_and_plot(model, loader, class_labels, dataset_name="Dataset"):

    model.eval()
    class_correct = {label: 0 for label in class_labels}
    class_total = {label: 0 for label in class_labels}

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for i in range(len(labels)):
                true_label = labels[i].item()
                predicted_label = preds[i].item()
                class_total[class_labels[true_label]] += 1
                if true_label == predicted_label:
                    class_correct[class_labels[true_label]] += 1

    class_accuracy = {
        label: (class_correct[label] / class_total[label] * 100) if class_total[label] > 0 else 0
        for label in class_labels
    }


    plt.figure(figsize=(8, 6))
    plt.bar(class_labels, class_accuracy.values(), color='skyblue')
    plt.title(f"Class-wise Accuracy for {dataset_name}")
    plt.xlabel("Class")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 100)
    for i, acc in enumerate(class_accuracy.values()):
        plt.text(i, acc + 2, f"{acc:.2f}%", ha='center', fontsize=10)
    plt.show()

    return class_accuracy

class_labels = ["Angular Leaf Spot", "Bean Rust", "Healthy"]

print("Class-wise Accuracy for Validation:")
validation_accuracy = calculate_class_accuracy_and_plot(model_mobileNet, val_loader, class_labels, dataset_name="Validation")

print("Class-wise Accuracy for Test:")
test_dataset = TestDataset(test_data, test_labels, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
test_accuracy = calculate_class_accuracy_and_plot(model_mobileNet, test_loader, class_labels, dataset_name="Test")


print("\nValidation Accuracy:")
for class_name, accuracy in validation_accuracy.items():
    print(f"{class_name}: {accuracy:.2f}%")

print("\nTest Accuracy:")
for class_name, accuracy in test_accuracy.items():
    print(f"{class_name}: {accuracy:.2f}%")


## 11. Optimizer comparison, tabulated

The frozen 3x3 grid from section 8, collected into one table.

In [ ]:
import pandas as pd

# Regenerated from the training logs above rather than transcribed by hand:
# four rows of the original table did not match what the runs printed, the
# worst being RMSProp/NasNet (recorded 90.52/87.97, logged 94.58/88.72).
# These are the final-epoch values of each run in section 8.
data = [
    {"Optimizer": "Adam",    "Model": "EfficientNetB6", "Tr-Acc(%)": 90.04, "Val-Acc(%)": 93.98, "Tr-Loss": 0.2849, "Val-Loss": 0.1481},
    {"Optimizer": "Adam",    "Model": "MobileNetV2",    "Tr-Acc(%)": 87.52, "Val-Acc(%)": 93.98, "Tr-Loss": 0.3264, "Val-Loss": 0.1573},
    {"Optimizer": "Adam",    "Model": "NasNet",         "Tr-Acc(%)": 95.36, "Val-Acc(%)": 89.47, "Tr-Loss": 0.1798, "Val-Loss": 0.2412},

    {"Optimizer": "RMSProp", "Model": "EfficientNetB6", "Tr-Acc(%)": 90.81, "Val-Acc(%)": 94.74, "Tr-Loss": 0.2453, "Val-Loss": 0.1359},
    {"Optimizer": "RMSProp", "Model": "MobileNetV2",    "Tr-Acc(%)": 87.52, "Val-Acc(%)": 87.97, "Tr-Loss": 0.3169, "Val-Loss": 0.2196},
    {"Optimizer": "RMSProp", "Model": "NasNet",         "Tr-Acc(%)": 94.58, "Val-Acc(%)": 88.72, "Tr-Loss": 0.1603, "Val-Loss": 0.2138},

    {"Optimizer": "Nadam",   "Model": "EfficientNetB6", "Tr-Acc(%)": 88.68, "Val-Acc(%)": 94.74, "Tr-Loss": 0.2886, "Val-Loss": 0.1489},
    {"Optimizer": "Nadam",   "Model": "MobileNetV2",    "Tr-Acc(%)": 87.43, "Val-Acc(%)": 90.23, "Tr-Loss": 0.2973, "Val-Loss": 0.1827},
    {"Optimizer": "Nadam",   "Model": "NasNet",         "Tr-Acc(%)": 94.49, "Val-Acc(%)": 87.97, "Tr-Loss": 0.1716, "Val-Loss": 0.2245},
]

df = pd.DataFrame(data)
print(df.to_string(index=False))